In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\huawei\AppData\Local\Temp\ipykernel_75144\3336491994.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\huawei\AppData\Local\Temp\ipykernel_75144\3336491994.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):


In [2]:
# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具

# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        }
    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()

# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.由于MCP的Tool是异步的，所以必须用ainvoke调用Agent，是异步调用
response = await agent.ainvoke(
    {"messages": [HumanMessage("现在是什么时间")]}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

现在是什么时间
================================== Ai Message ==================================
Tool Calls:
  current_time (call_00_FU4EHKgh4QEJveAAwXCR6138)
 Call ID: call_00_FU4EHKgh4QEJveAAwXCR6138
  Args:
    format: YYYY-MM-DD HH:mm:ss
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': 'Current UTC time is 2026-11-14 08:11:44, and the time in Asia/Shanghai is 2026-11-14 16:11:44.', 'id': 'lc_8af6938d-3a0c-4618-aa81-2ca34dc32545'}]
================================== Ai Message ==================================

现在是 **2026年11月14日 16:11:44**（北京时间，UTC+8）。

对应的 UTC 时间是 2026-11-14 08:11:44。需要其他时区的时间吗？
